# Per-class test accuracy of the saved (base) models

Evaluates each pretrained Baseline checkpoint in `saved_models/<arc>/best` on the **full held-out test set** (`data/cifar-10-batches-py/test_batch`, separate from the `data_batch_*` training files) and reports accuracy **per class, over all 10 classes**.

**Key point:** these checkpoints have an **8-output head**, so the model can only ever predict the 8 classes it was trained on. The 2 held-out classes have no output neuron — the model cannot predict them, so they score **~0%**. Evaluating over all 10 classes makes this visible (unlike an 8-class-only view, which hides the held-out pair).

Evidence from the data: with `TRAIN_CLASSES = [0..7]`, airplane/automobile score ~90/96% (clean, dedicated neurons) while ship/truck collapse to ~0% — so these models were trained on **classes 0–7 (airplane…horse), holding out 8=ship and 9=truck**. Ships/trucks get absorbed into their visual look-alikes (airplane/automobile).

**Assumptions to match training:** `TRAIN_CLASSES` (order == `--classes` at train time), `RELU = 2`, inputs `/255`. If unsure of the held-out pair, run the **diagnostic** cell — it finds them independently of `TRAIN_CLASSES`.

In [ ]:
import os, sys

# Make project root importable whether this notebook sits at the root or in ipynb_files/.
ROOT = os.getcwd()
if not os.path.exists(os.path.join(ROOT, 'funcs.py')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
os.chdir(ROOT)
print('Project root:', ROOT)

import pickle
import numpy as np
import pandas as pd
import torch

from funcs import load_batch
from py_files.Baseline import QResNet, BasicBlock

In [ ]:
# ---- configuration -------------------------------------------------
# The 8 classes each checkpoint was trained on, in the ORDER given to --classes.
# head logit i corresponds to original CIFAR class TRAIN_CLASSES[i].
# Determined from the data: trained on 0-7, held out 8=ship, 9=truck.
TRAIN_CLASSES = [0, 1, 2, 3, 4, 5, 6, 7]

RELU = 2  # Baseline checkpoints were trained with ReLU2

# saved_models/<folder> -> architecture (num_blocks). Folder name == first block count.
ARCHS = {
    '3': [3, 3, 3],
    '5': [5, 5, 5],
    '9': [9, 9, 9],
}
CKPT_TMPL = 'saved_models/{arc}/best'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

# CIFAR-10 class names (index = original label)
with open('data/cifar-10-batches-py/batches.meta', 'rb') as f:
    LABEL_NAMES = [n.decode() for n in pickle.load(f, encoding='bytes')[b'label_names']]
print('Classes:', LABEL_NAMES)
assert len(TRAIN_CLASSES) == 8, 'These checkpoints have an 8-class head.'
HELD_OUT = [c for c in range(10) if c not in TRAIN_CLASSES]
print('Held-out (should score ~0%):', [LABEL_NAMES[c] for c in HELD_OUT])

In [ ]:
# ---- load the FULL held-out test set (all 10 classes) ---------------
images_test, labels_test = load_batch('data/cifar-10-batches-py/test_batch')
X_test = torch.from_numpy(images_test).float() / 255.0  # normalise to [0, 1]
y_test = torch.from_numpy(labels_test).long()           # ORIGINAL labels 0..9
print('Test samples (all 10 classes):', len(X_test))

In [ ]:
# ---- evaluation helpers --------------------------------------------
TRAIN_CLASSES_ARR = np.array(TRAIN_CLASSES)

def load_model(num_blocks):
    """Build a Baseline QResNet (8-class head). No training wrapper -> CPU or GPU."""
    model = QResNet(RELU, BasicBlock, num_blocks, classes=TRAIN_CLASSES)
    return model.to(device).eval()

@torch.no_grad()
def predict_original(model, X, batch=512):
    """Predictions mapped to ORIGINAL CIFAR class indices (head argmax i -> TRAIN_CLASSES[i]).
    Held-out classes can never be produced, so they score 0 by construction."""
    preds = []
    for i in range(0, len(X), batch):
        idx = model(X[i:i+batch].to(device)).argmax(1).cpu().numpy()
        preds.append(TRAIN_CLASSES_ARR[idx])
    return np.concatenate(preds)

@torch.no_grad()
def head_index_counts(model, X, y, batch=512):
    """10x8 matrix: rows = true class (0..9), cols = predicted head index (0..7).
    Each TRAINED class concentrates in one column; held-out classes own none."""
    M = np.zeros((10, 8), dtype=int)
    for i in range(0, len(X), batch):
        idx = model(X[i:i+batch].to(device)).argmax(1).cpu().numpy()
        yb = y[i:i+batch].numpy()
        np.add.at(M, (yb, idx), 1)
    return M

In [ ]:
# ---- per-class accuracy over ALL 10 classes, every architecture -----
y_np = y_test.numpy()
results, overall = {}, {}

for arc, num_blocks in ARCHS.items():
    model = load_model(num_blocks)
    model.load_state_dict(torch.load(CKPT_TMPL.format(arc=arc), map_location=device, weights_only=True))

    pred = predict_original(model, X_test)
    acc = np.array([100.0 * (pred[y_np == c] == c).mean() for c in range(10)])
    col = f'arc {arc} ({"-".join(map(str, num_blocks))})'
    results[col] = pd.Series(acc, index=LABEL_NAMES)
    overall[col] = 100.0 * (pred == y_np).mean()
    print(f'{col}: overall {overall[col]:.2f}%')

In [ ]:
# ---- per-class table (rows = all 10 classes; held-out ~0%) ----------
table = pd.DataFrame(results)
table.loc['— OVERALL —'] = pd.Series(overall)
table.index.name = 'class'
held = {LABEL_NAMES[c]: '(held-out)' for c in HELD_OUT}
table.insert(0, 'note', [held.get(n, '') for n in table.index])
table.round(2)

In [ ]:
# ---- DIAGNOSTIC: which 8 classes does the head actually know? --------
# Independent of TRAIN_CLASSES. Rows = true class, cols = head index. A trained
# class puts almost all its mass in ONE column (its own neuron); the 2 held-out
# classes spread out -> lowest 'peak' fraction. This is the unambiguous answer.
diag_arc = '9'
model = load_model(ARCHS[diag_arc])
model.load_state_dict(torch.load(CKPT_TMPL.format(arc=diag_arc), map_location=device, weights_only=True))
M = head_index_counts(model, X_test, y_test)
frac = M / M.sum(1, keepdims=True)
diag = pd.DataFrame(frac, index=LABEL_NAMES, columns=[f'head{j}' for j in range(8)])
diag['peak'] = diag.max(1)
print('Two lowest-peak rows = held-out classes:',
      diag['peak'].sort_values().head(2).index.tolist())
print('Head index -> dominant true class:',
      {f'head{j}': LABEL_NAMES[int(M[:, j].argmax())] for j in range(8)})
diag.round(2)

In [ ]:
# ---- bar chart of per-class accuracy (held-out classes ~0) ----------
import matplotlib.pyplot as plt

ax = pd.DataFrame(results).plot.bar(figsize=(11, 5))
ax.set_ylabel('test accuracy (%)')
ax.set_title('Per-class test accuracy of saved base models (all 10 classes)')
ax.set_ylim(0, 100)
ax.legend(title='architecture')
plt.tight_layout()
plt.show()